# Build a VISTA2D Workflow 建立 VISTA2D 工作流程

## Check GPU Resources

Run only one GPU course at a time on this workshop VM.

1. **Save and close the previous kernel.** Save the previous course's notebook, then shut down its kernel in **Running Terminals and Kernels** in that course's JupyterLab. Closing a browser tab does not release GPU memory.
2. **Check GPU use.** Open **File → New → Terminal** and run `bootcamp-resources status`.
3. **Prepare this course.** If any BioNeMo models are running, run `bootcamp-resources stop --confirm`.
4. **Confirm readiness.** Run `bootcamp-resources status` again. Continue only when the BioNeMo models are stopped and `gpu_processes` is empty.

For the complete instructions, see `00_START_HERE.txt` in the course root directory.

**50-minute lab / 50 分鐘實作**

Your agent will create three scripts that are absent from this starter. You stay
in this notebook to inspect each file **before and after**, execute it, and reveal
VISTA2D's real microscopy predictions. Continue using your existing agent interface.

代理將建立起始教材沒有的三個腳本。你留在筆記本檢視每個檔案的**生成前後**、實際執行，
並展示 VISTA2D 的真實顯微影像預測。請沿用既有代理介面。

The instructor has prepared dependencies and staged upstream source. We provide
real images, official asset URLs, a short API brief, and a small mask writer.
The model runner and result visualization are yours to build with the agent.

講師已安裝相依套件並放好上游原始碼。提供真實影像、官方網址、簡短介面說明與小型遮罩儲存器。
模型執行程式與結果視覺化由你透過代理建立。

English and Traditional Chinese prompt versions are equivalent. Paste **only one
language section** into the agent, not a Python cell. Complete versions also live
under prompts/en/ and prompts/zh-TW/. Use the existing uv environment; do not create
one or install packages. Official weights are pre-cached on the VM image.
The text-only language model cannot see images;
you provide visual judgment. These examples are demonstrations, not held-out benchmarks.

中英文提示詞意義相同，請只選一種語言貼到代理，而非 Python 儲存格。兩種完整版本亦位於
prompts/en/ 與 prompts/zh-TW/。沿用既有 uv 環境，不建立環境或安裝套件；VM 映像已預放官方權重。
純文字語言模型不能看圖，由你判斷影像。
這些範例是示範，不是獨立保留的評測資料。


In [ ]:
from pathlib import Path
import difflib
import json
import os
import subprocess
import sys
import tempfile
from IPython.display import Code, Markdown, display
from PIL import Image

project_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "classroom.json").is_file()), None)
if project_root is None:
    raise RuntimeError("Open inside the lesson folder / 請在課程目錄開啟")
os.chdir(project_root)
classroom = json.loads(Path("classroom.json").read_text())
print("Project / 專案:", project_root)
print("Notebook Python:", sys.executable)
print("Model Python:", classroom["vista2d"]["python"])
print("Upstream config exists / 上游設定存在:", Path(classroom["vista2d"]["bundle"], "configs/inference.json").is_file())
snapshots = globals().get("snapshots", {})
SAMPLE = "fluorescence"
OUTPUT = Path("outputs/fluorescence")

def before_agent(filename):
    path = Path(filename)
    snapshots[filename] = path.read_text() if path.is_file() else ""
    display(Markdown(f"### Before the agent 代理執行前 — `{filename}`"))
    if path.is_file():
        display(Code(snapshots[filename], language="python"))
    else:
        print("File does not exist yet; the agent will create it. / 檔案尚未存在，接著由代理建立。")

def after_agent(filename):
    path = Path(filename)
    if filename not in snapshots:
        raise RuntimeError("Run the Before cell before asking the agent / 請先執行生成前儲存格")
    if not path.is_file():
        print("Ask the agent to create the file first / 請先讓代理建立檔案:", filename)
        return
    text = path.read_text()
    display(Markdown(f"### After the agent 代理執行後 — `{filename}`"))
    display(Code(text, language="python"))
    diff = "".join(difflib.unified_diff(snapshots[filename].splitlines(keepends=True),
                                     text.splitlines(keepends=True), fromfile="before/"+filename,
                                     tofile="after/"+filename))
    display(Markdown("**Code changes 程式差異**"))
    display(Code(diff or "No changes / 沒有變更", language="diff"))

def execute(*args, interpreter=None):
    command = [interpreter or classroom["vista2d"]["python"], *map(str, args)]
    print("Execute / 執行:", command)
    result = subprocess.run(command, cwd=project_root, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError("Execution failed. Share the error with your agent. / 執行失敗，請將錯誤提供給代理。")

def show_image(path):
    path = Path(path)
    if path.is_file():
        with Image.open(path) as image:
            display(image.copy())
    else:
        print("Not created yet / 尚未產生:", path)

show_image("data/microscopy/gallery.jpg")


## Mission 1 Create model preparation 任務一建立模型準備程式

**5–17 minutes / 第 5–17 分鐘**

Run the Before cell first. It prints the file like `cat`, or shows that it does
not yet exist. Then ask your agent to create the file using the prompt below.

先執行 Before 儲存格，如同 cat 印出檔案，或顯示尚未存在，再用下方提示詞要求代理建立。


In [ ]:
before_agent("prepare_model.py")


# Mission 1 Write the model preparation script 任務一撰寫模型準備程式

Copy only one language section into your agent.
請只選擇一種語言的段落貼給代理。

## English

Read WORKSHOP.md, docs/VISTA2D_API.md, and the checkpoints entries in assets.json.
Create prepare_model.py implementing that brief using Python's standard library.
It must download missing checkpoints, verify MD5, reuse valid cached files, preserve
mismatching existing files, support --check-only, and write preparation.json after success.
Do not call a finished workshop helper or copy instructor code. Do not install packages.
Run a syntax check and --help, then stop so I can inspect the code and run it in the notebook.

Reply in English.

Work from the lesson root and use the existing uv-managed environment. Use
`.venv/bin/python` for Python syntax checks and `--help`. For the model subprocess,
use the absolute interpreter already configured in `classroom.json`. Do not create
an environment, run `uv venv`, install or upgrade packages, or change GPU drivers.
The official checkpoints are pre-cached under `models/` in this VM image: verify
and reuse them. Never delete or force a re-download of valid cached weights.

Implement the missing-file download path, but do not download or execute preparation yourself. The notebook will use `prepare_model.py --check-only` to verify the pre-cached weights. Stop after syntax checking and `--help`.
## 繁體中文

請閱讀 WORKSHOP.md、docs/VISTA2D_API.md，以及 assets.json 中的 checkpoints。
依照介面說明，以 Python 標準函式庫建立 prepare_model.py。
程式必須下載缺少的權重、驗證 MD5、沿用有效快取、保留並回報校驗不符的既有檔案，
支援 --check-only，並且只在全部成功後寫入 preparation.json。
不要呼叫已完成的 workshop 工具，不要複製講師解答，也不要安裝套件。
請執行語法檢查與 --help，然後停止，讓我在筆記本檢視新程式並親自執行。
請以繁體中文說明你完成的工作。

請在教材根目錄工作，沿用已建立的 uv 虛擬環境，以 `.venv/bin/python` 執行語法檢查
與 `--help`；模型子程序使用 `classroom.json` 已設定的絕對 Python 路徑。
不要建立新環境、執行 `uv venv`、安裝或升級套件，也不要更動 GPU 驅動。
這個 VM 映像已在 `models/` 預放官方權重，請驗證後沿用，不要刪除或強制重新下載有效快取。

請實作缺漏檔案的下載流程，但不要自行下載或執行模型準備。筆記本會以 `prepare_model.py --check-only` 驗證預放權重。完成語法檢查與 `--help` 後即停止。

In [ ]:
after_agent("prepare_model.py")


### Execute preparation 執行準備

Inspect the new code before running this cell. A cached checkpoint is verified
and reused with --check-only; it must not be reported as a new download. This
classroom path makes no checkpoint downloads, even when a file is missing.

先檢視新程式再執行。使用 --check-only 驗證並沿用快取，不可稱為新下載；即使缺少權重，
這個課堂流程也不會啟動下載，而是回報問題。


In [ ]:
execute("prepare_model.py", "--check-only")
execute("tools/check_mission.py", "prepare")
print(Path("preparation.json").read_text())


**Explain / 說明:** What code did the agent write to verify downloads?
Why are SAM and VISTA2D checkpoints both present?

代理寫了哪些下載驗證程式？為什麼同時需要 SAM 與 VISTA2D 權重？

Answer / 回答：


## Mission 2 Create VISTA2D inference 任務二建立 VISTA2D 推論

**17–33 minutes / 第 17–33 分鐘**

We supply the official configuration and exact interface brief. Ask the agent
to write a runnable script that adapts the config and calls MONAI.

提供官方設定與介面說明，請代理撰寫可執行的腳本，調整設定並呼叫 MONAI。


In [ ]:
before_agent("run_vista.py")


# Mission 2 Write a VISTA2D runner 任務二撰寫 VISTA2D 執行程式

Copy only one language section into your agent.
請只選擇一種語言的段落貼給代理。

## English

Read docs/VISTA2D_API.md and vendor/VISTA/vista2d/configs/inference.json.
Create run_vista.py with --sample, --output, and --dry-run as specified in the brief.
Write a derived MONAI config, preserving preprocessing, network, inferer, and decoding.
Use the supplied support.save_mask.SaveInstanceMask only for saving the integer mask.
Execute the configured model Python through subprocess, preserve a log, and write run.json
only after actual successful inference. Do not train, change upstream source, import
workshop wrappers, or fabricate a mask. Syntax-check the file and test --help. Stop before
GPU execution: I will inspect the code and run its dry run and inference from the notebook.

Reply in English.

Work from the lesson root and use the existing uv-managed environment. Use
`.venv/bin/python` for Python syntax checks and `--help`. For the model subprocess,
use the absolute interpreter already configured in `classroom.json`. Do not create
an environment, run `uv venv`, install or upgrade packages, or change GPU drivers.
The official checkpoints are pre-cached under `models/` in this VM image: verify
and reuse them. Never delete or force a re-download of valid cached weights.

Do not execute `--dry-run` or real inference yourself, and do not create trial output directories. Test only syntax and `--help`; I will run both dry-run and inference in the notebook.
## 繁體中文

請閱讀 docs/VISTA2D_API.md 與 vendor/VISTA/vista2d/configs/inference.json。
依照介面說明，建立支援 --sample、--output、--dry-run 的 run_vista.py。
程式需產生衍生的 MONAI 設定，保留原本的前處理、網路、推論器及實例解碼。
只使用已提供的 support.save_mask.SaveInstanceMask 來儲存整數遮罩。
透過 subprocess 執行 classroom.json 指定的模型 Python，保留 inference.log，
而且只在真正推論成功後寫入 run.json。
不要訓練模型、更動上游原始碼、匯入 workshop 包裝程式，或製造假遮罩。
請檢查語法並執行 --help，然後停止，不要啟動 GPU 推論。
我會在筆記本檢視程式，再執行只產生設定的 --dry-run 與真正推論。
請以繁體中文回覆。

請在教材根目錄工作，沿用已建立的 uv 虛擬環境，以 `.venv/bin/python` 執行語法檢查
與 `--help`；模型子程序使用 `classroom.json` 已設定的絕對 Python 路徑。
不要建立新環境、執行 `uv venv`、安裝或升級套件，也不要更動 GPU 驅動。
這個 VM 映像已在 `models/` 預放官方權重，請驗證後沿用，不要刪除或強制重新下載有效快取。

請勿自行執行 `--dry-run` 或真正推論，也不要建立試跑輸出目錄。只檢查語法及 `--help`，由我在筆記本執行 dry-run 與推論。

In [ ]:
after_agent("run_vista.py")


### Inspect the generated configuration 檢查產生的設定

A dry run exercises the new script without executing the model. It must not
claim an inference result. The temporary directory is removed after inspection.

dry-run 執行新腳本但不啟動模型，不可聲稱完成推論。檢查後會移除暫存目錄。


In [ ]:
with tempfile.TemporaryDirectory(prefix="vista2d-config-") as temporary_output:
    execute("run_vista.py", "--sample", SAMPLE, "--output", temporary_output, "--dry-run")
    draft = json.loads((Path(temporary_output) / "vista_inference.json").read_text())
    print(json.dumps({key: draft[key] for key in ["input_dict", "sam_ckpt_path", "pretrained_ckpt_path", "inferer"]}, indent=2))


### Execute the real model 執行真正模型

This cell runs the agent-created script on the GPU. If OUTPUT already contains
results, choose a new directory in the setup cell. Keep the original run.

此儲存格在 GPU 執行代理撰寫的腳本。若 OUTPUT 已有結果，請在設定儲存格改新目錄，保留舊結果。


In [ ]:
execute("run_vista.py", "--sample", SAMPLE, "--output", OUTPUT)
execute("tools/check_mission.py", "infer", "--output", OUTPUT)
print((OUTPUT / "run.json").read_text())


## Mission 3 Create the visual reveal 任務三建立結果視覺化

**33–45 minutes / 第 33–45 分鐘**

Ask the agent to turn VISTA2D's integer mask into a side-by-side comparison.
The notebook only displays the PNG; the agent must write the boundary rendering.

請代理將 VISTA2D 整數遮罩轉為並排比較圖。筆記本只顯示 PNG，邊界繪製程式必須由代理撰寫。


In [ ]:
before_agent("visualize.py")


# Mission 3 Write the result visualization 任務三撰寫結果視覺化

Copy only one language section into your agent.
請只選擇一種語言的段落貼給代理。

## English

Read the visualize.py contract in docs/VISTA2D_API.md. Create visualize.py using Pillow.
Read the original image and the predicted integer TIFF mask, validate matching sizes,
and draw deterministic bright boundaries on a copy. Save the original and prediction
side by side as one comparison.png without resizing or changing the inputs. Provide
--image, --mask, and --output. Do not use a completed overlay helper or alter model results.
Syntax-check and test --help, then stop for notebook inspection and execution. You cannot
see the image: ask me to inspect the resulting comparison.

Reply in English.

Work from the lesson root and use the existing uv-managed environment. Use
`.venv/bin/python` for Python syntax checks and `--help`. For the model subprocess,
use the absolute interpreter already configured in `classroom.json`. Do not create
an environment, run `uv venv`, install or upgrade packages, or change GPU drivers.
The official checkpoints are pre-cached under `models/` in this VM image: verify
and reuse them. Never delete or force a re-download of valid cached weights.

Every positive label must use a bright color; never assign the black background color to a foreground boundary. The comparison canvas must be exactly twice the input width, without a horizontal gutter; an extra caption height is allowed.
## 繁體中文

請閱讀 docs/VISTA2D_API.md 中 visualize.py 的介面說明，以 Pillow 建立 visualize.py。
讀取原始影像與模型預測的整數 TIFF 遮罩，檢查尺寸一致，在副本上畫出固定且明亮的預測邊界。
將原圖與預測並排存成 comparison.png，不縮放，也不更動輸入影像或遮罩。
支援 --image、--mask、--output。不要使用已完成的疊圖工具，也不要改變模型結果。
請檢查語法並執行 --help，然後停止，讓我在筆記本檢視程式並親自執行。
你無法觀看影像，請提醒我觀察輸出的比較圖。請以繁體中文回覆。

請在教材根目錄工作，沿用已建立的 uv 虛擬環境，以 `.venv/bin/python` 執行語法檢查
與 `--help`；模型子程序使用 `classroom.json` 已設定的絕對 Python 路徑。
不要建立新環境、執行 `uv venv`、安裝或升級套件，也不要更動 GPU 驅動。
這個 VM 映像已在 `models/` 預放官方權重，請驗證後沿用，不要刪除或強制重新下載有效快取。

所有正標籤都必須使用明亮色彩，前景邊界不可使用黑色背景色。比較圖的寬度必須恰為輸入寬度的兩倍，中間不要留水平空隙；高度可另加標題區域。

In [ ]:
after_agent("visualize.py")


In [ ]:
record = json.loads((OUTPUT / "run.json").read_text())
execute("visualize.py", "--image", record["input_image"], "--mask", OUTPUT / "mask.tif",
        "--output", OUTPUT / "comparison.png")
execute("tools/check_mission.py", "visualize", "--output", OUTPUT)
show_image(OUTPUT / "comparison.png")


## Inspect and save 觀察與儲存

**45–50 minutes / 第 45–50 分鐘**

Which predicted boundary looks convincing? Which object might be merged, split,
or missed? Which useful script did your agent create? What did you ask it to repair?

哪個預測邊界可信？哪個物件可能合併、切碎或漏掉？代理建立了哪個有用腳本？你要求修復了什麼？

Answers / 回答：

Save the notebook with code/diff outputs, your scripts, comparison.png, run.json,
and inference.log. A file check does not establish biological accuracy. Record any
instructor checkpoint assistance rather than calling it agent-generated code.

儲存含程式／差異輸出的筆記本、腳本、比較圖及執行紀錄。檔案檢查不能證明生物學正確。
若使用講師解答，請註記協助，不要稱為代理生成。


## Optional Reuse the workflow 選做沿用流程

Only attempt this when all three scripts work. These cells execute the scripts
you just created on another real input. Nuclei-only input raises a different
target-definition question; its blue channel is from Cellpose img02.

三個腳本皆可執行後再選做。下方儲存格沿用剛建立的腳本處理另一張真實輸入。
細胞核輸入涉及不同目標定義，其藍色通道來自 Cellpose img02。

# Optional Reuse your new scripts 選做沿用新寫的程式

Copy only one language section into your agent.
請只選擇一種語言的段落貼給代理。

## English

Using the scripts you created, tell me the notebook commands for a second run with
--sample cellpose_demo and --output outputs/cellpose-demo, followed by visualization.
Do not retrain or rewrite the model. After that, optionally use the nuclei sample and
ask whether our intended target is a nucleus or a whole cell. Do not launch these runs
until I execute the notebook cell. Do not judge accuracy without labeled comparisons.

Reply in English.

Work from the lesson root and use the existing uv-managed environment. Use
`.venv/bin/python` for Python syntax checks and `--help`. For the model subprocess,
use the absolute interpreter already configured in `classroom.json`. Do not create
an environment, run `uv venv`, install or upgrade packages, or change GPU drivers.
The official checkpoints are pre-cached under `models/` in this VM image: verify
and reuse them. Never delete or force a re-download of valid cached weights.
## 繁體中文

請使用你剛建立的程式，告訴我在筆記本執行第二張影像推論及後續視覺化的命令。
推論使用 --sample cellpose_demo 與 --output outputs/cellpose-demo。
不要重新訓練或改寫模型。之後可以選做 nuclei，請提醒我思考預期分割的是細胞核還是完整細胞。
請等我執行筆記本儲存格後才啟動這些步驟，不要自行執行，也不要在缺少標註比較時判定準確率。
請以繁體中文回覆。

請在教材根目錄工作，沿用已建立的 uv 虛擬環境，以 `.venv/bin/python` 執行語法檢查
與 `--help`；模型子程序使用 `classroom.json` 已設定的絕對 Python 路徑。
不要建立新環境、執行 `uv venv`、安裝或升級套件，也不要更動 GPU 驅動。
這個 VM 映像已在 `models/` 預放官方權重，請驗證後沿用，不要刪除或強制重新下載有效快取。


In [ ]:
# Optional / 選做: reuse your new scripts on a second image.
SECOND_OUTPUT = Path("outputs/cellpose-demo")
execute("run_vista.py", "--sample", "cellpose_demo", "--output", SECOND_OUTPUT)
second_record = json.loads((SECOND_OUTPUT / "run.json").read_text())
execute("visualize.py", "--image", second_record["input_image"], "--mask", SECOND_OUTPUT / "mask.tif",
        "--output", SECOND_OUTPUT / "comparison.png")
show_image(SECOND_OUTPUT / "comparison.png")


## Save and Release GPU Resources

1. **Save and shut down.** Save this notebook, then shut down its kernel in **Running Terminals and Kernels**.
2. **Release resources.** In **File → New → Terminal**, run `bootcamp-resources status` to check that this course has released the GPU.
3. **Check before switching.** Run `bootcamp-resources status` and confirm that `gpu_processes` is empty before opening the next course's kernel.

When returning to BioNeMo, follow **Check GPU Resources** at the beginning of its lesson and wait for all three models to become **healthy**.
